# Healthcare: HL7 v2 and DICOM

Bedside monitors, analyzers and modalities talk **HL7 v2** (events and results over MLLP) and **DICOM** (images over C-STORE). Everything here is synthetic — fictional patients and schematic phantoms, never for clinical use.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.8.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.8.0-preview.1"
#r "nuget: IoTCom.Net.Adapters.Dicom, 0.8.0-preview.1"

## Build and parse an ORU^R01
One OBX segment per measurement, LOINC-coded.

In [ ]:
using IoTCom.Net.Protocols.Hl7;

var (patient, scenario, bed) = PatientMonitorSimulator.DemoWard[0];
var monitor = new PatientMonitorSimulator(patient, scenario, bed, onset: TimeSpan.Zero);
var oru = monitor.ToOru(monitor.Next(TimeSpan.FromMinutes(8), DateTimeOffset.Now));
Console.WriteLine(oru.Encode().Replace('\r', '\n'));
foreach (var o in Hl7Message.Parse(oru.Encode()).GetObservations())
    Console.WriteLine($"{o.Code.Text,-36} {o.Value,6} {o.Units,-8} {o.AbnormalFlag}");

## Send over MLLP and receive the ACK

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Transports;

var link = new InMemoryTransportListener();
await using var receiver = Hl7MllpServer.Create(o => o.ListenInMemory(link));
receiver.MessageReceived += (_, e) => Console.WriteLine($"received {e.Message.MessageType} {e.Message.ControlId}");
await receiver.StartAsync();
await using var sender = Hl7MllpClient.Create(o => o.UseInMemory(link));
var ack = await sender.SendAsync(oru);
Console.WriteLine($"ACK {ack.AckCode()} for {ack["MSA.2"]}");

## DICOM: a synthetic study over C-STORE, rendered with a window
The planted finding is the ground truth (useful for testing AI pipelines).

In [ ]:
using IoTCom.Net.Adapters.Dicom;

await using var pacs = DicomStoreServer.Create(o => o.Port = 11120);
await pacs.StartAsync();
DicomReceived? got = null;
pacs.ImageReceived += r => got = r;
await using var modality = DicomStoreClient.Create(o => o.Port = 11120);
await modality.EchoAsync();
var study = SyntheticImaging.Generate(SyntheticModality.ChestCt, SyntheticFinding.Pneumothorax);
await modality.StoreAsync(study.File);
var image = DicomRenderer.Render(got!.File.Dataset, DicomRenderer.Presets["CT lung"]);
Console.WriteLine($"{got.Modality} {image.Width}x{image.Height}, PNG {image.ToPng().Length / 1024} KB — planted: {study.FindingDescription}");

## Going further
The Gallery demos *ICU bedside monitors* and *Imaging AI pre-read* add NEWS2 scoring, trend analysis and an LLM (SBAR notes and vision pre-reads) on top of these building blocks — see `docs/en/guides/medical-ai.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*